In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT linux -> Test on linux
ADASYN Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "linux"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/adasyn/linux"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on ADASYN-oversampled source (all-except-linux)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (linux)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled linux
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (linux Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except linux -> Test on linux (ADASYN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/5] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (linux)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply ADASYN to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying ADASYN...")
    adasyn                        = ADASYN(random_state=SEED)
    train_emb_ada, train_lbl_ada  = adasyn.fit_resample(train_emb, train_lbl)
    train_emb_ada                 = train_emb_ada.astype(np.float32)
    train_lbl_ada                 = train_lbl_ada.astype(np.int32)
    print(f"      Before ADASYN : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After ADASYN  : {train_emb_ada.shape}  "
          f"vuln={train_lbl_ada.sum()}  benign={int((train_lbl_ada == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_ada, train_lbl_ada,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_ada, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_ada == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_ada, train_lbl_ada, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (linux Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"        : "adasyn",
        "train"             : "all_except_linux",
        "test"              : TEST_PROJ,
        "n_train_projects"  : int(len(set(proj_all[train_mask]))),
        "n_train_samples"   : int(len(train_lbl)),
        "n_train_adasyn"    : int(len(train_lbl_ada)),
        "n_test_samples"    : int(len(test_lbl)),
        "n_vuln_train"      : int(train_lbl.sum()),
        "n_vuln_train_adasyn": int(train_lbl_ada.sum()),
        "n_vuln_test"       : int(test_lbl.sum()),
        "accuracy"          : round(float(accuracy),  4),
        "precision"         : round(float(precision), 4),
        "recall"            : round(float(recall),    4),
        "f1"                : round(float(f1),        4),
        "auc"               : round(float(auc),        4),
        "g_mean"            : round(float(g_mean),    4),
        "pf"                : round(float(pf),        4),
        "confusion_matrix"  : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "adasyn_linux.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_adasyn_linux.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()


=== PrimeVul | Train on ALL except linux -> Test on linux (ADASYN) ===
    Device: mps

[1/5] Loading combined embeddings...
      Training pool             : (170682, 256)  vuln=4852  benign=165830
      Projects in training pool : 753
      Test set (linux)          : (53851, 256)  vuln=1152  benign=52699
      Training imbalance ratio  : 0.0284

[2/5] Normalizing embeddings...

[3/5] Applying ADASYN...
      Before ADASYN : (170682, 256)  vuln=4852  benign=165830
      After ADASYN  : (332479, 256)  vuln=166649  benign=165830

[4/5] Running ensemble iterations...


Ensemble:   0%|          | 0/3 [00:00<?, ?iter/s]


      [Iteration 1/3] Training neural network...


Iter 1 - Phase 1:   0%|          | 0/50 [00:00<?, ?it/s]

      Iter 1 - Phase 1 | Epoch 1/50 - val_loss: 0.4892
      Iter 1 - Phase 1 | Epoch 2/50 - val_loss: 0.4562
      Iter 1 - Phase 1 | Epoch 3/50 - val_loss: 0.4314
      Iter 1 - Phase 1 | Epoch 4/50 - val_loss: 0.4034
      Iter 1 - Phase 1 | Epoch 5/50 - val_loss: 0.3720
      Iter 1 - Phase 1 | Epoch 6/50 - val_loss: 0.3414
      Iter 1 - Phase 1 | Epoch 7/50 - val_loss: 0.3154
      Iter 1 - Phase 1 | Epoch 8/50 - val_loss: 0.2930
      Iter 1 - Phase 1 | Epoch 9/50 - val_loss: 0.2719
      Iter 1 - Phase 1 | Epoch 10/50 - val_loss: 0.2552
      Iter 1 - Phase 1 | Epoch 11/50 - val_loss: 0.2404
      Iter 1 - Phase 1 | Epoch 12/50 - val_loss: 0.2249
      Iter 1 - Phase 1 | Epoch 13/50 - val_loss: 0.2155
      Iter 1 - Phase 1 | Epoch 14/50 - val_loss: 0.2013
      Iter 1 - Phase 1 | Epoch 15/50 - val_loss: 0.1906
      Iter 1 - Phase 1 | Epoch 16/50 - val_loss: 0.1802
      Iter 1 - Phase 1 | Epoch 17/50 - val_loss: 0.1724
      Iter 1 - Phase 1 | Epoch 18/50 - val_loss: 0.1660
 

...

      Iter 1 - Phase 2 | Epoch 1/30 - val_loss: 0.0941
      Iter 1 - Phase 2 | Epoch 2/30 - val_loss: 0.0893
      Iter 1 - Phase 2 | Epoch 3/30 - val_loss: 0.0888
      Iter 1 - Phase 2 | Epoch 4/30 - val_loss: 0.0863
      Iter 1 - Phase 2 | Epoch 5/30 - val_loss: 0.0803
      Iter 1 - Phase 2 | Epoch 6/30 - val_loss: 0.0837
      Iter 1 - Phase 2 | Epoch 7/30 - val_loss: 0.0818
      Iter 1 - Phase 2 | Epoch 8/30 - val_loss: 0.0797
      Iter 1 - Phase 2 | Epoch 9/30 - val_loss: 0.0770
      Iter 1 - Phase 2 | Epoch 10/30 - val_loss: 0.0749
      Iter 1 - Phase 2 | Epoch 11/30 - val_loss: 0.0767
      Iter 1 - Phase 2 | Epoch 12/30 - val_loss: 0.0748
      Iter 1 - Phase 2 | Epoch 13/30 - val_loss: 0.0749
      Iter 1 - Phase 2 | Epoch 14/30 - val_loss: 0.0722
      Iter 1 - Phase 2 | Epoch 15/30 - val_loss: 0.0696
      Iter 1 - Phase 2 | Epoch 16/30 - val_loss: 0.0723
      Iter 1 - Phase 2 | Epoch 17/30 - val_loss: 0.0706
      Iter 1 - Phase 2 | Epoch 18/30 - val_loss: 0.0706
 

Iter 2 - Phase 1:   0%|          | 0/50 [00:00<?, ?it/s]

      Iter 2 - Phase 1 | Epoch 1/50 - val_loss: 0.4864
      Iter 2 - Phase 1 | Epoch 2/50 - val_loss: 0.4576
      Iter 2 - Phase 1 | Epoch 3/50 - val_loss: 0.4322
      Iter 2 - Phase 1 | Epoch 4/50 - val_loss: 0.4030
      Iter 2 - Phase 1 | Epoch 5/50 - val_loss: 0.3708
      Iter 2 - Phase 1 | Epoch 6/50 - val_loss: 0.3407
      Iter 2 - Phase 1 | Epoch 7/50 - val_loss: 0.3171
      Iter 2 - Phase 1 | Epoch 8/50 - val_loss: 0.2935
      Iter 2 - Phase 1 | Epoch 9/50 - val_loss: 0.2747
      Iter 2 - Phase 1 | Epoch 10/50 - val_loss: 0.2556
      Iter 2 - Phase 1 | Epoch 11/50 - val_loss: 0.2418
      Iter 2 - Phase 1 | Epoch 12/50 - val_loss: 0.2261
      Iter 2 - Phase 1 | Epoch 13/50 - val_loss: 0.2144
      Iter 2 - Phase 1 | Epoch 14/50 - val_loss: 0.2026
      Iter 2 - Phase 1 | Epoch 15/50 - val_loss: 0.1940
      Iter 2 - Phase 1 | Epoch 16/50 - val_loss: 0.1841
      Iter 2 - Phase 1 | Epoch 17/50 - val_loss: 0.1788
      Iter 2 - Phase 1 | Epoch 18/50 - val_loss: 0.1690
 

Iter 2 - Phase 2:   0%|          | 0/30 [00:00<?, ?it/s]

      Iter 2 - Phase 2 | Epoch 1/30 - val_loss: 0.1031
      Iter 2 - Phase 2 | Epoch 2/30 - val_loss: 0.0995


In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT Chrome -> Test on Chrome
ADASYN Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "Chrome"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/adasyn/Chrome"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on ADASYN-oversampled source (all-except-Chrome)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (Chrome)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled Chrome
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (Chrome Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except Chrome -> Test on Chrome (ADASYN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/5] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (Chrome)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply ADASYN to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying ADASYN...")
    adasyn                        = ADASYN(random_state=SEED)
    train_emb_ada, train_lbl_ada  = adasyn.fit_resample(train_emb, train_lbl)
    train_emb_ada                 = train_emb_ada.astype(np.float32)
    train_lbl_ada                 = train_lbl_ada.astype(np.int32)
    print(f"      Before ADASYN : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After ADASYN  : {train_emb_ada.shape}  "
          f"vuln={train_lbl_ada.sum()}  benign={int((train_lbl_ada == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_ada, train_lbl_ada,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_ada, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_ada == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_ada, train_lbl_ada, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (Chrome Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"        : "adasyn",
        "train"             : "all_except_Chrome",
        "test"              : TEST_PROJ,
        "n_train_projects"  : int(len(set(proj_all[train_mask]))),
        "n_train_samples"   : int(len(train_lbl)),
        "n_train_adasyn"    : int(len(train_lbl_ada)),
        "n_test_samples"    : int(len(test_lbl)),
        "n_vuln_train"      : int(train_lbl.sum()),
        "n_vuln_train_adasyn": int(train_lbl_ada.sum()),
        "n_vuln_test"       : int(test_lbl.sum()),
        "accuracy"          : round(float(accuracy),  4),
        "precision"         : round(float(precision), 4),
        "recall"            : round(float(recall),    4),
        "f1"                : round(float(f1),        4),
        "auc"               : round(float(auc),        4),
        "g_mean"            : round(float(g_mean),    4),
        "pf"                : round(float(pf),        4),
        "confusion_matrix"  : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "adasyn_Chrome.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_adasyn_Chrome.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT qemu -> Test on qemu
ADASYN Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "qemu"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/adasyn/qemu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on ADASYN-oversampled source (all-except-qemu)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (qemu)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled qemu
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (qemu Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except qemu -> Test on qemu (ADASYN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/5] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (qemu)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply ADASYN to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying ADASYN...")
    adasyn                        = ADASYN(random_state=SEED)
    train_emb_ada, train_lbl_ada  = adasyn.fit_resample(train_emb, train_lbl)
    train_emb_ada                 = train_emb_ada.astype(np.float32)
    train_lbl_ada                 = train_lbl_ada.astype(np.int32)
    print(f"      Before ADASYN : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After ADASYN  : {train_emb_ada.shape}  "
          f"vuln={train_lbl_ada.sum()}  benign={int((train_lbl_ada == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_ada, train_lbl_ada,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_ada, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_ada == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_ada, train_lbl_ada, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (qemu Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"        : "adasyn",
        "train"             : "all_except_qemu",
        "test"              : TEST_PROJ,
        "n_train_projects"  : int(len(set(proj_all[train_mask]))),
        "n_train_samples"   : int(len(train_lbl)),
        "n_train_adasyn"    : int(len(train_lbl_ada)),
        "n_test_samples"    : int(len(test_lbl)),
        "n_vuln_train"      : int(train_lbl.sum()),
        "n_vuln_train_adasyn": int(train_lbl_ada.sum()),
        "n_vuln_test"       : int(test_lbl.sum()),
        "accuracy"          : round(float(accuracy),  4),
        "precision"         : round(float(precision), 4),
        "recall"            : round(float(recall),    4),
        "f1"                : round(float(f1),        4),
        "auc"               : round(float(auc),        4),
        "g_mean"            : round(float(g_mean),    4),
        "pf"                : round(float(pf),        4),
        "confusion_matrix"  : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "adasyn_qemu.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_adasyn_qemu.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT gpac -> Test on gpac
ADASYN Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "gpac"
COMBINED_FILE = "../../../../embedding/primevul/codet5/primevul_embedded_codet5.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/adasyn/gpac"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on ADASYN-oversampled source (all-except-gpac)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (gpac)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled gpac
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (gpac Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except gpac -> Test on gpac (ADASYN) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load combined embeddings and split by project
    # ----------------------------------------------------------
    print("\n[1/5] Loading combined embeddings...")

    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (gpac)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply ADASYN to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying ADASYN...")
    adasyn                        = ADASYN(random_state=SEED)
    train_emb_ada, train_lbl_ada  = adasyn.fit_resample(train_emb, train_lbl)
    train_emb_ada                 = train_emb_ada.astype(np.float32)
    train_lbl_ada                 = train_lbl_ada.astype(np.int32)
    print(f"      Before ADASYN : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After ADASYN  : {train_emb_ada.shape}  "
          f"vuln={train_lbl_ada.sum()}  benign={int((train_lbl_ada == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_ada, train_lbl_ada,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_ada, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_ada == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_ada, train_lbl_ada, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (gpac Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"        : "adasyn",
        "train"             : "all_except_gpac",
        "test"              : TEST_PROJ,
        "n_train_projects"  : int(len(set(proj_all[train_mask]))),
        "n_train_samples"   : int(len(train_lbl)),
        "n_train_adasyn"    : int(len(train_lbl_ada)),
        "n_test_samples"    : int(len(test_lbl)),
        "n_vuln_train"      : int(train_lbl.sum()),
        "n_vuln_train_adasyn": int(train_lbl_ada.sum()),
        "n_vuln_test"       : int(test_lbl.sum()),
        "accuracy"          : round(float(accuracy),  4),
        "precision"         : round(float(precision), 4),
        "recall"            : round(float(recall),    4),
        "f1"                : round(float(f1),        4),
        "auc"               : round(float(auc),        4),
        "g_mean"            : round(float(g_mean),    4),
        "pf"                : round(float(pf),        4),
        "confusion_matrix"  : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "adasyn_gpac.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_adasyn_gpac.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()